In [5]:
pip install pypdf sentence-transformers faiss-cpu ollama langchain langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [6]:
import time
import urllib.request
import os
import json
import numpy as np
import faiss
import ollama
from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Timing helper ──────────────────────────────────────
class Timer:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, *args):
        self.elapsed = time.time() - self.start
        print(f"  ✓ {self.label}: {self.elapsed:.2f}s")

print("All imports successful")

All imports successful


In [7]:
PDF_URL  = "https://arxiv.org/pdf/2309.15217"
PDF_PATH = Path("../data/poc_ragas_paper.pdf")
PDF_PATH.parent.mkdir(parents=True, exist_ok=True)

if PDF_PATH.exists():
    print(f"PDF already exists: {PDF_PATH}")
else:
    print("Downloading RAGAS paper from arXiv...")
    with Timer("Download"):
        urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f"Saved to: {PDF_PATH}")

print(f"File size: {PDF_PATH.stat().st_size / 1024:.1f} KB")

PDF already exists: ../data/poc_ragas_paper.pdf
File size: 226.5 KB


In [8]:
print("Extracting text from PDF...")

with Timer("PDF extraction"):
    reader   = PdfReader(PDF_PATH)
    pages    = [page.extract_text() for page in reader.pages]
    pages    = [p for p in pages if p and len(p.strip()) > 50]
    raw_text = "\n\n".join(pages)

print(f"Pages extracted:    {len(pages)}")
print(f"Total characters:   {len(raw_text):,}")
print(f"Total words (est.): {len(raw_text.split()):,}")
print()
print("=== FIRST 500 CHARS ===")
print(raw_text[:500])

Extracting text from PDF...
  ✓ PDF extraction: 0.12s
Pages extracted:    8
Total characters:   31,955
Total words (est.): 4,923

=== FIRST 500 CHARS ===
Ragas: Automated Evaluation of Retrieval Augmented Generation
Shahul Es†, Jithin James †, Luis Espinosa-Anke ∗♢, Steven Schockaert ∗
†Exploding Gradients
∗CardiffNLP, Cardiff University, United Kingdom
♢AMPLYFI, United Kingdom
shahules786@gmail.com,jamesjithin97@gmail.com
{espinosa-ankel,schockaerts1}@cardiff.ac.uk
Abstract
We introduce Ragas (Retrieval Augmented
Generation Assessment), a framework for
reference-free evaluation of Retrieval Aug-
mented Generation (RAG) pipelines. RAG
systems are


In [9]:
print("Chunking with fixed 256-token window (V0 strategy)...")

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 256,
    chunk_overlap = 32,
    length_function = len,
)

with Timer("Chunking"):
    chunks = splitter.split_text(raw_text)

# Build chunk dicts matching V0 schema
chunk_dicts = [
    {
        "chunk_id"    : f"poc_chunk_{i:04d}",
        "paper_id"    : "ragas_2309.15217",
        "text"        : chunk,
        "token_count" : len(chunk.split()),
        "char_count"  : len(chunk),
    }
    for i, chunk in enumerate(chunks)
]

print(f"Total chunks:       {len(chunk_dicts)}")
print(f"Avg tokens/chunk:   {np.mean([c['token_count'] for c in chunk_dicts]):.1f}")
print(f"Min tokens/chunk:   {min(c['token_count'] for c in chunk_dicts)}")
print(f"Max tokens/chunk:   {max(c['token_count'] for c in chunk_dicts)}")
print()
print("=== FIRST CHUNK ===")
print(chunk_dicts[0]['text'])

Chunking with fixed 256-token window (V0 strategy)...
  ✓ Chunking: 0.00s
Total chunks:       142
Avg tokens/chunk:   35.0
Min tokens/chunk:   5
Max tokens/chunk:   46

=== FIRST CHUNK ===
Ragas: Automated Evaluation of Retrieval Augmented Generation
Shahul Es†, Jithin James †, Luis Espinosa-Anke ∗♢, Steven Schockaert ∗
†Exploding Gradients
∗CardiffNLP, Cardiff University, United Kingdom
♢AMPLYFI, United Kingdom


In [10]:
print("Loading all-MiniLM-L6-v2 embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["text"] for c in chunk_dicts]

print(f"Embedding {len(texts)} chunks...")

with Timer("Embedding") as t:
    embeddings = model.encode(
        texts,
        batch_size       = 32,
        show_progress_bar= True,
        normalize_embeddings = True   # L2 normalise = cosine similarity
    )

embeddings = np.array(embeddings).astype("float32")

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Expected shape:         ({len(texts)}, 384)")
print(f"Memory usage (est.):    {embeddings.nbytes / 1024:.1f} KB")

# ── Record for supervisor report ──────────────────────
embedding_time = t.elapsed
chunks_per_sec = len(texts) / embedding_time
print(f"\n── METRIC 1: Embedding time: {embedding_time:.2f}s")
print(f"── METRIC 1: Throughput:     {chunks_per_sec:.1f} chunks/sec")

Loading all-MiniLM-L6-v2 embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 142 chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ Embedding: 1.31s

Embedding matrix shape: (142, 384)
Expected shape:         (142, 384)
Memory usage (est.):    213.0 KB

── METRIC 1: Embedding time: 1.31s
── METRIC 1: Throughput:     108.8 chunks/sec


In [11]:
print("Building FAISS IndexFlatL2...")

dimension = embeddings.shape[1]  # 384

with Timer("FAISS index build") as t:
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

print(f"Index type:     IndexFlatL2")
print(f"Dimension:      {dimension}")
print(f"Vectors stored: {index.ntotal}")
print(f"── METRIC: Index build time: {t.elapsed:.3f}s")

Building FAISS IndexFlatL2...
  ✓ FAISS index build: 0.00s
Index type:     IndexFlatL2
Dimension:      384
Vectors stored: 142
── METRIC: Index build time: 0.002s


In [12]:
TEST_QUERIES = [
    "What metrics does RAGAS use to evaluate RAG pipelines?",
    "How is context recall computed in RAGAS?",
    "What datasets were used to evaluate RAGAS?",
]

K = 5
retrieval_times = []

print(f"Running {len(TEST_QUERIES)} test queries, top-K={K}\n")

for query in TEST_QUERIES:
    with Timer(f"Retrieval: '{query[:50]}...'") as t:
        query_embedding = model.encode(
            [query],
            normalize_embeddings=True
        ).astype("float32")
        distances, indices = index.search(query_embedding, K)
    
    retrieval_times.append(t.elapsed)
    
    print(f"\n  Top chunk retrieved:")
    print(f"  {chunk_dicts[indices[0][0]]['text'][:200]}...")

avg_retrieval = np.mean(retrieval_times)
print(f"\n── METRIC 2: Avg retrieval latency: {avg_retrieval*1000:.2f}ms")

Running 3 test queries, top-K=5

  ✓ Retrieval: 'What metrics does RAGAS use to evaluate RAG pipeli...': 0.11s

  Top chunk retrieved:
  identify relevant and focused context passages,
the ability of the LLM to exploit such passages
in a faithful way, or the quality of the genera-
tion itself. With Ragas, we put forward a suite
of metr...
  ✓ Retrieval: 'How is context recall computed in RAGAS?...': 0.03s

  Top chunk retrieved:
  c(q) and then uses the retrieved context to generate
an answer as(q). When building a RAG system,...
  ✓ Retrieval: 'What datasets were used to evaluate RAGAS?...': 0.03s

  Top chunk retrieved:
  Ragas1, a framework for the automated assessment
1Ragas is available at https://github.com/
explodinggradients/ragas.
arXiv:2309.15217v2  [cs.CL]  28 Apr 2025...

── METRIC 2: Avg retrieval latency: 57.77ms


In [13]:
import ollama

r = ollama.chat(model="llama3.2:3b", messages=[{"role":"user","content":"say: OK"}])
print(r["message"]["content"])

OK. Is there something I can help you with?


In [28]:
TEST_QUERY = "What metrics does RAGAS use to evaluate RAG pipelines?"

print(f"Running end-to-end pipeline for:\n'{TEST_QUERY}'\n")

e2e_start = time.time()

# Step 1 — Retrieve
query_embedding = model.encode(
    [TEST_QUERY],
    normalize_embeddings=True
).astype("float32")
distances, indices = index.search(query_embedding, K)
retrieved_chunks = [chunk_dicts[i]["text"] for i in indices[0]]

# Step 2 — Build prompt
context = "\n\n---\n\n".join(retrieved_chunks)
prompt  = f"""You are a helpful assistant. Answer the question using only 
the context provided below. If the context does not contain the answer, 
say "I cannot find this in the provided context."

CONTEXT:
{context}

QUESTION:
{TEST_QUERY}

ANSWER:"""

# Step 3 — Generate with Llama 3.2 3B
print("Sending to Llama 3.2 3B via Ollama...")
with Timer("Ollama generation") as gen_timer:
    response = ollama.chat(
        model   = "llama3.2:3b",
        messages= [{"role": "user", "content": prompt}],
        options = {"temperature": 0}
    )

answer      = response["message"]["content"]
e2e_elapsed = time.time() - e2e_start

# Token usage
prompt_tokens = response.get("prompt_eval_count", "N/A")
gen_tokens    = response.get("eval_count", "N/A")

print(f"\n=== GENERATED ANSWER ===")
print(answer)
print(f"\n── METRIC 3: Generation time:        {gen_timer.elapsed:.2f}s")
print(f"── METRIC 3: End-to-end time:         {e2e_elapsed:.2f}s")
print(f"── METRIC 4: Prompt tokens:           {prompt_tokens}")
print(f"── METRIC 4: Generated tokens:        {gen_tokens}")

Running end-to-end pipeline for:
'What metrics does RAGAS use to evaluate RAG pipelines?'

Sending to Llama 3.2 3B via Ollama...
  ✓ Ollama generation: 6.94s

=== GENERATED ANSWER ===
The context mentions that "With Ragas, we put forward a suite of metrics which can be used to evaluate these [RAG] pipelines." However, it does not specify what those metrics are.

── METRIC 3: Generation time:        6.94s
── METRIC 3: End-to-end time:         7.90s
── METRIC 4: Prompt tokens:           335
── METRIC 4: Generated tokens:        41


In [15]:
print("=" * 55)
print("  POC RESULTS — FOR SUPERVISOR REPORT")
print("=" * 55)
print()
print(f"  Document:        RAGAS paper (arXiv 2309.15217)")
print(f"  Pages:           {len(pages)}")
print(f"  Total chunks:    {len(chunk_dicts)}")
print(f"  Chunk strategy:  Fixed 256-token window, 32 overlap")
print(f"  Embedding model: all-MiniLM-L6-v2 (384-dim)")
print(f"  LLM:             Llama 3.2 3B via Ollama (temp=0)")
print()
print(f"  METRIC 1 — Embedding time:       {embedding_time:.2f}s")
print(f"             Throughput:            {chunks_per_sec:.1f} chunks/sec")
print()
print(f"  METRIC 2 — Avg retrieval latency:{avg_retrieval*1000:.2f}ms")
print()
print(f"  METRIC 3 — Generation time:      {gen_timer.elapsed:.2f}s")
print(f"             End-to-end per query:  {e2e_elapsed:.2f}s")
print()
print(f"  METRIC 4 — Prompt tokens:        {prompt_tokens}")
print(f"             Generated tokens:      {gen_tokens}")
print()
print(f"  FEASIBILITY: 150-query estimate")
est_150 = e2e_elapsed * 150
print(f"  @ {e2e_elapsed:.1f}s/query × 150 = {est_150/60:.1f} minutes")
print("=" * 55)

  POC RESULTS — FOR SUPERVISOR REPORT

  Document:        RAGAS paper (arXiv 2309.15217)
  Pages:           8
  Total chunks:    142
  Chunk strategy:  Fixed 256-token window, 32 overlap
  Embedding model: all-MiniLM-L6-v2 (384-dim)
  LLM:             Llama 3.2 3B via Ollama (temp=0)

  METRIC 1 — Embedding time:       1.31s
             Throughput:            108.8 chunks/sec

  METRIC 2 — Avg retrieval latency:57.77ms

  METRIC 3 — Generation time:      3.08s
             End-to-end per query:  3.41s

  METRIC 4 — Prompt tokens:        335
             Generated tokens:      41

  FEASIBILITY: 150-query estimate
  @ 3.4s/query × 150 = 8.5 minutes
